In [32]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)
print("src exists:", (PROJECT_ROOT / "src").exists())

Project root: /Users/zaidashraf/Quantum-Secured-Threat-Intelligence-Pipeline
src exists: True


In [33]:
import importlib
import src.ner

importlib.reload(src.ner)

from src.ner import (
    extract_entities,
    load_attack_dictionary,
)

In [34]:
ATTACK_ENTITY_PATH = (
    "../data/processed/attack_entities.csv"
)

attack_dictionary = load_attack_dictionary(
    ATTACK_ENTITY_PATH
)

print("Dictionary entries:", len(attack_dictionary))

Dictionary entries: 2403


In [35]:
from pathlib import Path
import pandas as pd

DATA_PATH = Path("../data/processed/processed_attack_dataset.csv")

df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
df.head()

Shape: (19745, 10)


,id,type,name,description,aliases,external_id,platforms,kill_chain_phases,created,modified
0,malware--007b44b6-e4c5-480b-b5b9-56f2081b1b7b,malware,HDoor,HDoor is malware that has been customized and ...,"[""HDoor"", ""Custom HDoor""]",S0061,"[""Windows""]",[],2017-05-31T21:32:40.801Z,2025-04-16T20:37:51.573Z
1,malware--00806466-754d-44ea-ad6f-0caf59cb8556,malware,TrickBot,TrickBot is a Trojan spyware program written i...,"[""TrickBot"", ""Totbrick"", ""TSPY_TRICKLOAD""]",S0266,"[""Windows""]",[],2018-10-17T00:14:20.652Z,2024-04-10T22:28:21.746Z
2,malware--00936d7a-451d-4a9c-88fc-05b141aa2d3f,malware,cd00r,cd00r is an open-source backdoor for UNIX and ...,"[""cd00r""]",S1204,"[""Network Devices""]",[],2025-02-19T20:07:23.223Z,2025-04-15T19:46:33.009Z
3,malware--00c3bfcb-99bd-4767-8c03-b08f585f5c8a,malware,PowerDuke,PowerDuke is a backdoor that was used by APT29...,"[""PowerDuke""]",S0139,"[""Windows""]",[],2017-05-31T21:33:19.746Z,2025-04-25T14:42:30.325Z
4,malware--00e7d565-9883-4ee5-b642-8fd17fd6a3f5,malware,EKANS,EKANS is ransomware variant written in Golang ...,"[""EKANS"", ""SNAKEHOSE""]",S0605,"[""Windows""]",[],2021-02-12T20:07:42.883Z,2025-04-16T20:37:51.908Z


In [2]:
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 19745 entries, 0 to 19744
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   id                 19745 non-null  str  
 1   type               19745 non-null  str  
 2   name               1748 non-null   str  
 3   description        19744 non-null  str  
 4   aliases            19745 non-null  str  
 5   external_id        1748 non-null   str  
 6   platforms          19745 non-null  str  
 7   kill_chain_phases  19745 non-null  str  
 8   created            19745 non-null  str  
 9   modified           19745 non-null  str  
dtypes: str(10)
memory usage: 6.5 MB


In [3]:
df["type"].value_counts()

type
relationship      17997
malware             726
attack-pattern      697
intrusion-set       174
tool                 95
campaign             56
Name: count, dtype: int64

In [4]:
df.isna().sum()

id                       0
type                     0
name                 17997
description              1
aliases                  0
external_id          17997
platforms                0
kill_chain_phases        0
created                  0
modified                 0
dtype: int64

In [5]:
entity_df = df[df["type"] != "relationship"].copy()
relationship_df = df[df["type"] == "relationship"].copy()

print("Entity records:", len(entity_df))
print("Relationship records:", len(relationship_df))

Entity records: 1748
Relationship records: 17997


In [6]:
ENTITY_PATH = Path("../data/processed/attack_entities.csv")

entity_df.to_csv(ENTITY_PATH, index=False)

print("Saved:", ENTITY_PATH)

Saved: ../data/processed/attack_entities.csv


In [7]:
threat_actors = entity_df[
    entity_df["type"] == "intrusion-set"
]

threat_actors[
    ["external_id", "name", "aliases", "description"]
].head(10)

,external_id,name,aliases,description
1518,G0119,Indrik Spider,"[""Indrik Spider"", ""Evil Corp"", ""Manatee Tempes...",Indrik Spider is a Russia-based cybercriminal ...
1520,G1014,LuminousMoth,"[""LuminousMoth""]",LuminousMoth is a Chinese-speaking cyber espio...
1521,G1051,Medusa Group,"[""Medusa Group""]",Medusa Group has been active since at least 20...
1523,G0102,Wizard Spider,"[""Wizard Spider"", ""UNC1878"", ""TEMP.MixMaster"",...",Wizard Spider is a Russia-based financially mo...
1524,G0066,Elderwood,"[""Elderwood"", ""Elderwood Gang"", ""Beijing Group...",Elderwood is a suspected Chinese cyber espiona...
1525,G0046,FIN7,"[""FIN7"", ""GOLD NIAGARA"", ""ITG14"", ""Carbon Spid...",FIN7 is a financially-motivated threat group t...
1526,G1048,UNC3886,"[""UNC3886""]",UNC3886 is a China-nexus cyberespionage group ...
1527,G1047,Velvet Ant,"[""Velvet Ant""]",Velvet Ant is a threat actor operating since a...
1529,G0090,WIRTE,"[""WIRTE"", ""Ashen Lepus""]","WIRTE is a cyberespionage actor, believed to b..."
1530,G1033,Star Blizzard,"[""Star Blizzard"", ""SEABORGIUM"", ""Callisto Grou...",Star Blizzard is a cyber espionage and influen...


In [8]:
malware = entity_df[
    entity_df["type"] == "malware"
]

malware[
    ["external_id", "name", "aliases", "description"]
].head(10)

,external_id,name,aliases,description
0,S0061,HDoor,"[""HDoor"", ""Custom HDoor""]",HDoor is malware that has been customized and ...
1,S0266,TrickBot,"[""TrickBot"", ""Totbrick"", ""TSPY_TRICKLOAD""]",TrickBot is a Trojan spyware program written i...
2,S1204,cd00r,"[""cd00r""]",cd00r is an open-source backdoor for UNIX and ...
3,S0139,PowerDuke,"[""PowerDuke""]",PowerDuke is a backdoor that was used by APT29...
4,S0605,EKANS,"[""EKANS"", ""SNAKEHOSE""]",EKANS is ransomware variant written in Golang ...
5,S0520,BLINDINGCAN,"[""BLINDINGCAN""]",BLINDINGCAN is a remote access Trojan that has...
6,S1100,Ninja,"[""Ninja""]",Ninja is a malware developed in C++ that has b...
7,S1145,Pikabot,"[""Pikabot""]",Pikabot is a backdoor used for initial access ...
8,S0206,Wiarp,"[""Wiarp""]",Wiarp is a trojan used by Elderwood to open a ...
9,S0662,RCSession,"[""RCSession""]",RCSession is a backdoor written in C++ that ha...


In [9]:
techniques = entity_df[
    entity_df["type"] == "attack-pattern"
]

techniques[
    [
        "external_id",
        "name",
        "kill_chain_phases",
        "platforms",
        "description",
    ]
].head(10)


,external_id,name,kill_chain_phases,platforms,description
821,T1055.011,Extra Window Memory Injection,"[""stealth"", ""privilege-escalation""]","[""Windows""]",Adversaries may inject malicious code into pro...
822,T1053.005,Scheduled Task,"[""execution"", ""persistence"", ""privilege-escala...","[""Windows""]",Adversaries may abuse the Windows Task Schedul...
823,T1205.002,Socket Filters,"[""stealth"", ""persistence"", ""command-and-control""]","[""Linux"", ""macOS"", ""Windows""]",Adversaries may attach filters to a network so...
824,T1560.001,Archive via Utility,"[""collection""]","[""Linux"", ""macOS"", ""Windows""]",Adversaries may use utilities to compress and/...
825,T1021.005,VNC,"[""lateral-movement""]","[""Linux"", ""Windows"", ""macOS""]",Adversaries may use Valid Accounts to remotely...
826,T1047,Windows Management Instrumentation,"[""execution""]","[""Windows""]",Adversaries may abuse Windows Management Instr...
827,T1687,Exploitation for Defense Impairment,"[""defense-impairment""]","[""IaaS"", ""Linux"", ""macOS"", ""SaaS"", ""Windows""]",Adversaries may exploit vulnerabilities in sec...
828,T1113,Screen Capture,"[""collection""]","[""Linux"", ""macOS"", ""Windows""]",Adversaries may attempt to take screen capture...
829,T1027.011,Fileless Storage,"[""stealth""]","[""Linux"", ""Windows""]","Adversaries may store data in ""fileless"" forma..."
830,T1037,Boot or Logon Initialization Scripts,"[""persistence"", ""privilege-escalation""]","[""ESXi"", ""Linux"", ""macOS"", ""Network Devices"", ...",Adversaries may use scripts automatically exec...


In [10]:
print("Duplicate IDs:", entity_df["id"].duplicated().sum())
print("Missing names:", entity_df["name"].isna().sum())

empty_descriptions = (
    entity_df["description"]
    .fillna("")
    .str.strip()
    .eq("")
    .sum()
)

print("Empty descriptions:", empty_descriptions)

Duplicate IDs: 0
Missing names: 0
Empty descriptions: 0


In [13]:
from src.ner import extract_entities

sample_text = """
APT29 used PowerShell and exploited CVE-2023-38831.
The activity mapped to T1059.001 and contacted 192.168.1.25.
"""

extract_entities(sample_text)

{'cves': ['CVE-2023-38831'],
 'attack_ids': ['T1059.001'],
 'ip_addresses': ['192.168.1.25'],
 'urls': [],
 'hashes': []}

In [30]:
from pathlib import Path

REPORT_PATH = Path(
    "../data/raw/sample_reports/testing/report_001.txt"
)

report_text = REPORT_PATH.read_text(encoding="utf-8")

print(report_text)

APT29 used PowerShell to execute malicious commands and exploit
CVE-2023-38831. The activity was mapped to T1059.001 and contacted
192.168.1.25 through https://malicious-example.com/payload.

The downloaded file had the SHA-256 hash
0123456789abcdef0123456789abcdef0123456789abcdef0123456789abcdef.


In [37]:
result = extract_entities(report_text)
result

{'cves': ['CVE-2023-38831'],
 'attack_ids': ['T1059.001'],
 'ip_addresses': ['192.168.1.25'],
 'urls': ['https://malicious-example.com/payload'],
 'hashes': ['0123456789abcdef0123456789abcdef0123456789abcdef0123456789abcdef'],
 'attack_entities': [],
 'categorized_entities': {'threat_actors': [],
  'malware': [],
  'tools': [],
  'techniques': [],
  'campaigns': [],
  'other': []}}

In [36]:
attack_dictionary = load_attack_dictionary(
    "../data/processed/attack_entities.csv"
)

result = extract_entities(
    report_text,
    attack_dictionary=attack_dictionary,
)

result

{'cves': ['CVE-2023-38831'],
 'attack_ids': ['T1059.001'],
 'ip_addresses': ['192.168.1.25'],
 'urls': ['https://malicious-example.com/payload'],
 'hashes': ['0123456789abcdef0123456789abcdef0123456789abcdef0123456789abcdef'],
 'attack_entities': [{'name': 'PowerShell',
   'type': 'attack-pattern',
   'external_id': 'T1059.001',
   'matched_text': 'PowerShell'},
  {'name': 'APT29',
   'type': 'intrusion-set',
   'external_id': 'G0016',
   'matched_alias': 'APT29',
   'matched_text': 'APT29'}],
 'categorized_entities': {'threat_actors': [{'name': 'APT29',
    'type': 'intrusion-set',
    'external_id': 'G0016',
    'matched_alias': 'APT29',
    'matched_text': 'APT29'}],
  'malware': [],
  'tools': [],
  'techniques': [{'name': 'PowerShell',
    'type': 'attack-pattern',
    'external_id': 'T1059.001',
    'matched_text': 'PowerShell'}],
  'campaigns': [],
  'other': []}}